<a href="https://colab.research.google.com/github/dvarelaj/nlp-miniproyecto-icesi/blob/main/Sesion%203/Mini_Proyecto_3__Transformers_arXiv_completado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller 3: Clasificación de Texto con Transformers
## Corpus: Resúmenes de Artículos Científicos (arXiv)

---

### Contexto del taller
Este notebook es la continuación directa de nuestra entrega anterior (el Taller 2). Como ya probamos modelos clásicos (Random Forest), Embeddings simples y redes recurrentes (LSTM) usando nuestro dataset de arXiv, ahora decidimos dar el salto a la arquitectura que cambió las reglas del NLP: los **Transformers** (basados en el paper *"Attention is All You Need"* de Vaswani et al.).

Para este reto cambiamos de entorno. En vez de usar **TensorFlow/Keras** como la vez pasada, nos pasamos a **PyTorch y PyTorch Lightning**, que es lo que más se está usando ahora mismo en investigación. Además, en lugar de simplemente importar un modelo ya hecho, armamos el Transformer **desde cero** (sus bloques de Positional Encodings, Multi-Head Attention y el TransformerBlock) para entender bien qué es lo que hace por debajo antes de tratarlo como una caja negra.

**Nuestra pregunta central:** ¿Será que un Transformer entrenado desde cero logra ganarle a los modelos que hicimos en el Taller 2 usando exactamente los mismos datos científicos?

---

## ¿Qué herramientas usamos?

* **PyTorch:** Lo usamos para armar los bloques del Transformer a mano (`nn.Module`), controlando nosotros mismos las operaciones matemáticas del mecanismo de atención.
* **PyTorch Lightning:** Nos sirvió para organizar el entrenamiento de una forma más limpia (es como el `.fit()` que usábamos en Keras, pero nos da más flexibilidad para experimentar).
* **HuggingFace Transformers:** De aquí sacamos el tokenizador `bert-base-uncased`. A diferencia del tokenizador sencillo por palabras que usamos antes, este usa sub-palabras (BPE), lo que nos ayuda un montón a manejar las palabras raras y la jerga científica de los abstracts.
* **Scikit-Learn:** Lo reciclamos para convertir las categorías a números (`LabelEncoder`), partir los datos de forma balanceada y calcular los `class_weights` para manejar el desbalance de las clases.
* **NLTK y Regex:** Mantuvimos nuestra misma función de limpieza del Taller 2 para quitarle a los textos todo ese ruido de comandos LaTeX y fórmulas matemáticas.
* **Matplotlib y Seaborn:** Para graficar cómo aprende el modelo, visualizar el Positional Encoding y analizar la matriz de confusión al final.

## Paso 1 — Instalación de dependencias y reproducibilidad

Como Google Colab ya trae PyTorch configurado con soporte para GPU, en esta primera celda solo instalamos las librerías extra que necesitamos (como `pytorch-lightning` y `transformers`).

Un detalle metodológico clave que aplicamos aquí fue fijar una **semilla aleatoria global** (`seed=42`), usando exactamente la misma que en nuestro Taller 2. Si dejábamos esto al azar, el *accuracy* final nos iba a cambiar en cada ejecución y la comparación entre modelos no sería justa. Esto pasa por tres fuentes de aleatoriedad en las redes neuronales:

* La **inicialización aleatoria** de los millones de pesos matemáticos al crear el modelo.
* El **orden (shuffle)** en el que los datos entran a la red en cada época de entrenamiento.
* El **Dropout**, que apaga neuronas al azar para evitar que el modelo se memorice los textos.

Al dejar esta semilla fija, eliminamos esas variaciones y garantizamos que cualquier persona que corra nuestro notebook va a obtener exactamente los mismos resultados.

In [ ]:
# Instalamos las librerías de Deep Learning que nos faltan en Colab
!pip install -q lightning==2.2.0 'transformers[torch]'==4.41.2 torchmetrics datasets

import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. SEMILLA GLOBAL (Para que los resultados sean reproducibles)
# ==========================================
import random
import numpy as np
import torch

def set_seed(seed: int = 42):
    # Fijamos la misma semilla que usamos en el taller anterior (42)
    # Esto es vital para que las variaciones en el accuracy no dependan del azar
    # y podamos comparar el Transformer con la LSTM de forma justa.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Obligamos a la GPU a realizar operaciones matemáticas deterministas
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Semilla global configurada en: {seed}")

set_seed(42)
print("Dependencias listas y entorno configurado.")

## Paso 2 — Importación de librerías y configuración del entorno

Para esta fase del taller configuramos un entorno que combina **PyTorch** (para toda la construcción de Deep Learning) y **HuggingFace** (para el manejo avanzado de texto).

Para aprovechar lo que ya nos había funcionado en el Taller 2, seguimos usando **NLTK** para limpiar las stopwords, **Pandas y NumPy** para manipular nuestro dataset de arXiv, y **Scikit-Learn** para codificar las etiquetas y calcular los pesos de clase (algo que fue clave para manejar el desbalance en nuestra entrega anterior).

Toda la arquitectura del Transformer la construimos a mano usando **PyTorch puro**, mientras que le dejamos a **PyTorch Lightning** la tarea de gestionar el ciclo de entrenamiento para mantener el notebook organizado. Por último, agregamos una validación para confirmar que Colab nos haya asignado la GPU correctamente antes de arrancar.

In [ ]:
# CONFIGURACIÓN DEL ENTORNO

# --- Librerías ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time
import requests
import xml.etree.ElementTree as ET
from tqdm.auto import tqdm

# --- (NLP) ---
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
# Descargamos los recursos de NLTK
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

# --- Machine Learning (Scikit-Learn) ---
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# --- Deep Learning (PyTorch) ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# --- PyTorch Lightning ---
from lightning import LightningModule, Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from torchmetrics import Accuracy

# --- HuggingFace ---
from transformers import AutoTokenizer

# --- Visualización ---
from wordcloud import WordCloud

# Verificamos que Colab nos haya asignado la GPU correctamente antes de arrancar
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("¡Librerías importadas con éxito!")
print(f"Dispositivo de entrenamiento: {device.upper()}")
if device == 'cuda':
    print(f"Tarjeta gráfica asignada: {torch.cuda.get_device_name(0)}")

## Paso 3 — Carga y Estructura del Dataset

Para esta fase, decidimos reutilizar exactamente el mismo dataset que extrajimos en el Taller 2 (los 20,000 registros descargados de la API de arXiv desde el 2022). Hicimos esto a propósito para garantizar que cualquier diferencia en los resultados finales se deba exclusivamente a la nueva arquitectura del Transformer y no a que hayamos cambiado los datos.

**Las columnas que nos interesan para el análisis son:**

* `abstract`: El texto del resumen científico (nuestra variable independiente, X).
* `primary_category`: La categoría o tema principal del artículo (lo que queremos predecir, y).
* `title`: El título (lo traemos como contexto para nosotros, pero no lo metemos a la red neuronal).

Tal como notamos en la entrega pasada, estos abstracts vienen con los mismos retos de siempre: mucha sintaxis de LaTeX, fórmulas matemáticas y una jerga súper técnica. Por eso, nos toca aplicarles la misma limpieza rigurosa antes de pasarlos por el modelo.

> **Nota sobre la carga de datos:** Como descargar todos los registros por la API desde cero toma casi 20 minutos, para este notebook exportamos el DataFrame limpio del Taller 2 a un `.csv` y lo cargamos directamente aquí. Así optimizamos el tiempo de ejecución si se necesita correr el notebook desde el principio.

In [ ]:
# EXTRACCIÓN DE DATOS (API de arXiv)

# Usamos exactamente los mismos parámetros de nuestro Taller 2
OAI_ENDPOINT = 'http://export.arxiv.org/oai2'
METADATA_PREFIX = 'arXiv'
MAX_RECORDS = 20000
BATCH_SIZE = 200
FROM_DATE = '2022-01-01'
SLEEP_BETWEEN = 1.0 # Pausa obligatoria para que la API de arXiv no nos bloquee la IP

ns = {
    'oai': 'http://www.openarchives.org/OAI/2.0/',
    'arxiv': 'http://arxiv.org/OAI/arXiv/'
}

def build_params(verb='ListRecords', metadataPrefix=METADATA_PREFIX,
                 resumptionToken=None, from_date=None):
    # Armamos los parámetros para la petición GET
    params = {'verb': verb}
    if resumptionToken:
        params['resumptionToken'] = resumptionToken
    else:
        params['metadataPrefix'] = metadataPrefix
        if from_date:
            params['from'] = from_date
    return params

def parse_records(xml_text):
    # Parseamos el XML que devuelve la API
    root = ET.fromstring(xml_text)
    records = []

    # Iteramos sobre cada artículo descargado
    for record in root.findall('.//oai:record', ns):
        try:
            meta = record.find('oai:metadata/arxiv:arXiv', ns)
            if meta is None:
                continue

            # Nos interesan solo estos tres campos para el modelo
            abstract = meta.findtext('arxiv:abstract', default='', namespaces=ns).strip()
            title    = meta.findtext('arxiv:title',    default='', namespaces=ns).strip()
            cats     = meta.findtext('arxiv:categories', default='', namespaces=ns).strip()

            # Como algunos artículos tienen varias categorías, nos quedamos con la principal
            primary  = cats.split()[0] if cats else ''

            if abstract and primary:
                records.append({'title': title, 'abstract': abstract,
                                'primary_category': primary})
        except Exception:
            # Si algún registro viene dañado, lo saltamos para no romper el código
            continue

    # Guardamos el token para poder pedir la siguiente página de resultados
    token_el = root.find('.//oai:resumptionToken', ns)
    token = token_el.text if token_el is not None else None

    return records, token

def fetch_arxiv_dataset(max_records=MAX_RECORDS, from_date=FROM_DATE):
    # Loop principal de descarga
    all_records = []
    token = None
    pbar = tqdm(total=max_records, desc='Descargando abstracts de arXiv')

    while len(all_records) < max_records:
        params = build_params(resumptionToken=token, from_date=from_date)
        response = requests.get(OAI_ENDPOINT, params=params, timeout=60)

        # Si la API se cae o nos rechaza la conexión, detenemos el loop
        if response.status_code != 200:
            print("Error en la conexión con la API de arXiv.")
            break

        records, token = parse_records(response.text)
        all_records.extend(records)
        pbar.update(len(records))

        if not token:
            break

        # Respetamos el tiempo de espera entre peticiones
        time.sleep(SLEEP_BETWEEN)

    pbar.close()
    return pd.DataFrame(all_records[:max_records])

print("Funciones de extracción listas. Podemos proceder a descargar o cargar el CSV.")

In [ ]:
# Ejecutamos la extracción de datos (o cargamos desde el CSV si lo exportamos antes)
# Nota: Si esto tarda mucho, recuerda que podemos cargar el CSV del taller anterior
df_arxiv = fetch_arxiv_dataset()

print(f"Total de abstracts recuperados: {df_arxiv.shape[0]} registros.")
print("Vistazo a los primeros 3 artículos descargados:")
df_arxiv.head(3)

## Paso 4 — Preprocesamiento de Texto

Para esta etapa, decidimos traernos tal cual el pipeline de limpieza que armamos en el Taller 2. Hacer esto es clave porque nos permite comparar de forma justa el rendimiento de este Transformer contra los modelos anteriores, asegurándonos de que todos están leyendo exactamente el mismo texto procesado.

**Las acciones de limpieza que aplicamos fueron:**

* **Limpieza especializada:** Eliminamos todos los comandos de LaTeX (`\comando`) y las fórmulas matemáticas que están entre `$...$`. Como notamos en la entrega pasada, esto es puro ruido que no le aporta valor semántico al clasificador. También filtramos las *stopwords* en inglés y borramos las palabras de menos de 3 letras.
* **Filtro del Top 10 de categorías:** Como el dataset original tiene 153 categorías diferentes (muchas con poquísimos datos), filtramos el corpus para quedarnos únicamente con las 10 más frecuentes. Así garantizamos que el modelo tenga suficientes ejemplos para aprender y mantenemos las mismas reglas de juego del Taller 2.
* **Codificación de etiquetas:** Transformamos las categorías de texto (por ejemplo, `cs.LG`) a números usando el `LabelEncoder` de Scikit-Learn, ya que PyTorch necesita tensores numéricos para poder entrenar.
* **Split estratificado 80/20:** Dividimos los datos dejando el 80% para entrenar y el 20% para probar. Usamos una división estratificada para asegurar que la proporción de cada categoría se mantenga igual en ambos conjuntos, y volvimos a usar nuestro `random_state=42` para que la partición sea siempre la misma.

In [ ]:
# Cargamos la lista de stopwords en inglés
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Reutilizamos la función del Taller 2 para limpiar el ruido específico de arXiv
    text = text.lower()

    # Eliminamos comandos de LaTeX y fórmulas matemáticas que no le aportan valor semántico al Transformer
    text = re.sub(r'\$.*?\$', '', text)
    text = re.sub(r'\\(\w+)', '', text)

    # Conservamos únicamente letras (quitamos números y signos de puntuación)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenizamos y filtramos las stopwords y palabras muy cortas (de 1 o 2 letras)
    tokenizer_re = RegexpTokenizer(r'\w+')
    tokens = tokenizer_re.tokenize(text)
    filtered = [w for w in tokens if w not in stop_words and len(w) > 2]

    return ' '.join(filtered)

# Nos quedamos solo con las 10 categorías principales para mitigar el desbalance extremo
category_counts = df_arxiv['primary_category'].value_counts()
top_10 = category_counts.head(10).index
df_filtered = df_arxiv[df_arxiv['primary_category'].isin(top_10)].copy()

# Aplicamos la limpieza a todo el dataframe
print("Limpiando los abstracts... esto toma unos segunditos.")
df_filtered['cleaned_text'] = df_filtered['abstract'].apply(clean_text)

# Convertimos las categorías de texto a índices numéricos para que PyTorch las entienda
le = LabelEncoder()
df_filtered['label'] = le.fit_transform(df_filtered['primary_category'])
num_classes = len(le.classes_)

print(f"¡Dataset listo! Nos quedamos con {df_filtered.shape[0]} registros y {num_classes} categorías.")
print(f"Categorías finales: {list(le.classes_)}")

In [ ]:
# Dividimos los datos dejando un 80% para entrenamiento y 20% para validación.
# Es vital usar stratify y el mismo random_state=42 para replicar exactamente
# las particiones que usamos con la LSTM y hacer una comparación justa.
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df_filtered['cleaned_text'].values,
    df_filtered['label'].values,
    test_size=0.2,
    random_state=42,
    stratify=df_filtered['label']
)

print(f"Partición completada -> Textos para entrenar: {len(X_train_text)} | Textos para prueba: {len(X_test_text)}")

## Paso 5 — Tokenizador HuggingFace (BERT)

En la entrega pasada usamos el `Tokenizer` básico de Keras, que simplemente le asigna un número único a cada palabra completa que encuentra. Pero para este notebook decidimos dar un paso más allá e implementar el tokenizador de `bert-base-uncased`. La gran ventaja de este es que usa **BPE (Byte Pair Encoding)**: en lugar de tomar palabras enteras, divide el texto en "sub-palabras" o fragmentos frecuentes.

Nos dimos cuenta de que este cambio es fundamental al trabajar con textos científicos. Por ejemplo, con el tokenizador de Keras, términos como *"astrophysical"* y *"astrophysics"* eran vistos por nuestra red anterior como dos palabras completamente distintas y sin relación. Con BPE, el modelo identifica que ambas comparten el fragmento `"astro"`, lo que le ayuda a entender más rápido que pertenecen al mismo campo de estudio sin importar el sufijo.

Además, el vocabulario de este tokenizador (que tiene 30,522 tokens) ya fue pre-entrenado leyendo toda la Wikipedia y el BookCorpus en inglés. Esto significa que ya trae "de fábrica" un conocimiento base sobre mucha de la terminología técnica que nos viene perfecto para analizar los papers de arXiv.

In [ ]:
# Apagamos esta variable de entorno para evitar unos warnings molestos
# de paralelismo que tira HuggingFace cuando corre en Colab
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Instanciamos el tokenizador de BERT que elegimos para este taller
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Hicimos una prueba rápida con una frase técnica para ver el BPE en acción
ejemplo = 'astrophysical observations using quantum entanglement'
tokens = tokenizer.tokenize(ejemplo)

print(f'Frase de prueba: "{ejemplo}"')
print(f'Así lo divide en sub-palabras (tokens): {tokens}')
print(f'Y así lo traduce a números (IDs) para la red: {tokenizer.encode(ejemplo)}')
print(f'\nTamaño de nuestro diccionario base: {tokenizer.vocab_size} tokens')

## Paso 6 — Dataset y DataLoaders de PyTorch

En el taller anterior con Keras, pasarle los datos al modelo era súper directo; solo le metíamos los arrays al `.fit()` y listo. Al pasarnos a PyTorch para armar este Transformer, nos dimos cuenta de que el control de la información es mucho más manual y explícito. Tuvimos que estructurar la carga de datos en dos piezas fundamentales:

* **La clase `Dataset`:** Aquí definimos las reglas de cómo PyTorch debe procesar un solo abstract a la vez. En esta clase configuramos que el texto se tokenice en el mismo momento en que se lee (on-the-fly) y que devuelva los tensores listos junto con su etiqueta numérica.
* **El `DataLoader`:** Es el administrador. Coge nuestro `Dataset` y se encarga de agrupar los textos en bloques (batches), revolverlos (shuffle) en cada época para que el modelo no se memorice el orden, y pasarlos eficientemente a la memoria.

**El detalle clave: la `attention_mask`**
Como los resúmenes de arXiv tienen diferentes tamaños, nos toca rellenar los más cortos con tokens vacíos (padding) para que todas las secuencias queden del mismo largo y no se rompan las matrices. El problema es que no queremos que el Transformer considere ese relleno como parte del contexto.

La `attention_mask` nos resuelve este problema: es un vector binario donde un `1` le indica al modelo "esta es una palabra real, préstale atención", y un `0` significa "esto es puro relleno, ignóralo". Así, el mecanismo de atención le pone un peso matemáticamente nulo al padding antes de hacer la clasificación.

In [ ]:
# Decidimos fijar el tamaño máximo de la secuencia en 256.
# Como analizamos antes, los abstracts de arXiv suelen tener entre 150 y 200 palabras,
# así que con 256 cubrimos la gran mayoría de los textos sin desperdiciar memoria RAM.
MAX_SEQUENCE_LENGTH = 256

class ArxivDataset(Dataset):
    # Clase personalizada para indicarle a PyTorch cómo leer y procesar nuestros textos
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        # PyTorch necesita saber cuántos registros tenemos en total para armar las épocas
        return len(self.texts)

    def __getitem__(self, idx):
        # Aquí tokenizamos el texto "al vuelo" cada vez que la red pide un dato.
        # Si el abstract es muy largo lo cortamos (truncation), y si es muy corto
        # lo rellenamos (padding) para que todos los tensores midan exactamente max_len.
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt'  # Pedimos que nos devuelva tensores de PyTorch directamente
        )

        # El squeeze(0) lo usamos para quitarle una dimensión extra (batch) que HuggingFace
        # le agrega por defecto, ya que el DataLoader se va a encargar de agrupar los batches después.
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Instanciamos nuestros datasets de entrenamiento y prueba
train_dataset = ArxivDataset(X_train_text, y_train, tokenizer, MAX_SEQUENCE_LENGTH)
test_dataset  = ArxivDataset(X_test_text,  y_test,  tokenizer, MAX_SEQUENCE_LENGTH)

# Hacemos una prueba rápida con el primer dato (sanity check) para confirmar que las dimensiones cuadran
sample = train_dataset[0]
print(f"Dimensiones de los IDs de entrada: {sample['input_ids'].shape}")
print(f"Dimensiones de la máscara de atención: {sample['attention_mask'].shape}")
print(f"Categoría (Label): {sample['label']} -> Que en texto original es: {le.inverse_transform([sample['label'].item()])[0]}")

In [ ]:
# Definimos el tamaño del batch en 32.
# A diferencia de la LSTM del Taller 2, notamos que los Transformers consumen
# muchísima más memoria de la GPU. Usar un batch de 32 nos pareció el punto ideal
# para aprovechar la tarjeta gráfica de Colab sin que se nos crashee por falta de memoria (OOM).
BATCH_SIZE_DL = 32

# El DataLoader de entrenamiento lleva shuffle=True para que en cada época
# el modelo vea los abstracts en un orden distinto y evitemos que memorice el dataset.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE_DL,
    shuffle=True,
    num_workers=2 # Usamos 2 workers para paralelizar la lectura de datos en la CPU
)

# El DataLoader de prueba no necesita shuffle porque solo lo usamos para evaluar al final.
test_loader  = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE_DL,
    shuffle=False,
    num_workers=2
)

print("DataLoaders empaquetados y listos para pasarle a la red.")
print(f"Total de batches (paquetes de 32 textos) para entrenar: {len(train_loader)}")
print(f"Total de batches para evaluar: {len(test_loader)}")

## Paso 7 — Positional Encodings

En el Taller 2, cuando armamos la red LSTM, no tuvimos que preocuparnos por el orden de las palabras porque esa arquitectura lee el texto de forma secuencial. El orden temporal venía "de fábrica": la palabra en el paso `t` siempre se procesa después de la del paso `t-1`.

Pero al pasarnos a los Transformers la historia cambia. Como estos modelos leen todas las palabras al mismo tiempo (en paralelo) para aprovechar al máximo la velocidad de la GPU, pierden por completo la noción de secuencia. Si le pasamos el texto así nomás, para la red sería exactamente lo mismo leer *"el modelo no es bueno"* que *"el modelo es no bueno"*.



Para solucionar este problema de amnesia posicional, implementamos la solución propuesta en el paper original: inyectarle a cada token información sobre su posición sumándole un vector matemático único justo antes de que entre al bloque de atención. Estas son las fórmulas que usamos:

$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right) \quad PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Donde `pos` representa la posición de la palabra en la frase e `i` es la dimensión del embedding. Usar estas ondas de seno y coseno es clave porque nos garantizan que cada posición tenga una huella matemática única y, al mismo tiempo, permite que las palabras que están cerca en el texto tengan representaciones similares.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    """Aquí implementamos matemáticamente la inyección de posición según el paper original."""

    def __init__(self, max_len: int, d_model: int):
        super().__init__()

        # Creamos el vector que representa la posición de la palabra (0, 1, 2...)
        pos = torch.arange(max_len).unsqueeze(1)          # (max_len, 1)
        # Creamos el vector para iterar sobre las dimensiones del embedding
        i = torch.arange(d_model).unsqueeze(0)            # (1, d_model)

        # Calculamos el denominador complejo de la fórmula (usando la base de 10,000)
        div_term = 1 / torch.pow(10000, (2 * (i // 2)) / d_model)
        angles = pos * div_term                           # (max_len, d_model)

        # Generamos la matriz vacía y le aplicamos seno a las columnas pares y coseno a las impares
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(angles[:, 0::2])
        pe[:, 1::2] = torch.cos(angles[:, 1::2])

        # Lo guardamos como un 'buffer'. Esto es crucial para decirle a PyTorch que
        # esta matriz es una constante matemática fija y no un peso que deba aprender/entrenar.
        self.register_buffer('pos_encoding', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # Al tensor de embeddings de entrada (x) le sumamos nuestra matriz posicional
        # x shape: (batch, seq_len, d_model)
        return x + self.pos_encoding[:, :x.size(1), :]


# Quisimos graficar una porción de esta matriz para asegurarnos de que nuestra
# implementación matemática estuviera arrojando los resultados esperados.
pe_viz = SinusoidalPositionalEncoding(max_len=256, d_model=128)
pe_matrix = pe_viz.pos_encoding.squeeze(0).numpy()

plt.figure(figsize=(12, 5))
plt.pcolormesh(pe_matrix[:80, :64], cmap='RdBu')
plt.xlabel('Dimensión del Embedding')
plt.ylabel('Posición en la Secuencia')
plt.title('Positional Encodings — cada fila es la "huella" de una posición')
plt.colorbar()
plt.tight_layout()
plt.show()

print("En la gráfica podemos confirmar el comportamiento esperado: cada fila (posición) tiene un patrón de colores único. ¡El modelo ya no sufrirá de amnesia posicional!")

### Token + Positional Embeddings

Para armar la entrada final que va a procesar nuestra red, decidimos encapsular el proceso en un solo bloque que combina dos elementos fundamentales:

1. **Token Embedding:** Es la capa de embeddings clásica (muy parecida a la que usamos en el taller 2) que traduce el número del token a un vector denso de tamaño `d_model`. Estos pesos sí los va a ir aprendiendo y ajustando el modelo a medida que entrena.
2. **Positional Encoding:** Aquí es donde le inyectamos la matriz matemática que graficamos arriba, sumándole a cada palabra su "coordenada" exacta.

El resultado final de esta suma es clave para que la arquitectura funcione: logramos que cada token no solo entienda *qué* significa en términos semánticos, sino exactamente *dónde está* ubicado dentro del abstract del artículo.

In [ ]:
class TokenAndPositionalEmbedding(nn.Module):
    """En esta clase unimos las dos piezas: el significado semántico de la palabra y su posición exacta."""

    def __init__(self, max_len: int, d_model: int, vocab_size: int, dropout: float = 0.1):
        super().__init__()
        # Capa de embedding tradicional. Le pasamos padding_idx=0 para que
        # el modelo no pierda tiempo aprendiendo una representación para el token de relleno.
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)

        # Instanciamos nuestra clase matemática de posiciones
        self.pos_enc   = SinusoidalPositionalEncoding(max_len, d_model)

        # Agregamos Dropout desde esta capa por recomendación del evaluador en el taller pasado,
        # así empezamos a regularizar la red lo más pronto posible para evitar sobreajuste.
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (batch, seq_len) — Son los IDs numéricos de los tokens
        emb = self.token_emb(x)   # Convertimos IDs a vectores densos: (batch, seq_len, d_model)
        emb = self.pos_enc(emb)   # Aquí le sumamos la matriz matemática de posición
        return self.dropout(emb)


# Antes de armar toda la red compleja, quisimos hacer un "sanity check" rápido
# con un batch inventado para asegurarnos de que las dimensiones de los tensores cuadraran bien.
vocab_size = tokenizer.vocab_size
d_model    = 128
tpe_test   = TokenAndPositionalEmbedding(MAX_SEQUENCE_LENGTH, d_model, vocab_size)

# Simulamos un batch falso con 2 secuencias de 256 tokens cada una
dummy_ids  = torch.randint(0, vocab_size, (2, MAX_SEQUENCE_LENGTH))
output     = tpe_test(dummy_ids)

print(f"Dimensiones de la entrada de prueba (IDs): {dummy_ids.shape}")
print(f"Dimensiones de la salida (Embeddings + Posición): {output.shape}")
print("¡Las dimensiones cruzan perfecto! Podemos avanzar tranquilos al mecanismo de atención.")

## Paso 8 — Multi-Head Attention

Este es el "corazón" de nuestro modelo y lo que realmente lo separa de la LSTM que usamos en el Taller 2. Mientras que la red recurrente tenía que memorizar lo que leyó antes, la **Atención** le permite al modelo preguntarse: *"Para entender esta palabra técnica en este abstract, ¿en qué otras partes del texto debería fijarme?"*

La operación que implementamos es el **Scaled Dot-Product Attention**, que usa tres conceptos clave:
* **Q (Query):** La palabra que estamos analizando en el momento.
* **K (Key):** Las demás palabras del texto que sirven como etiquetas de búsqueda.
* **V (Value):** El contenido real o significado que cada palabra aporta.

La fórmula que programamos es:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Al multiplicar $Q$ por $K^T$, el modelo calcula qué tanta relación hay entre cada par de palabras. Luego, el softmax convierte eso en porcentajes (pesos de atención) y el resultado final nos da un vector que ya tiene "masticado" el contexto completo del abstract.

**¿Para qué usamos múltiples cabezas?**
En lugar de hacer este proceso una sola vez, decidimos proyectar la información en varias "cabezas" independientes que trabajan en paralelo. La idea es que cada una se especialice: una cabeza puede aprender a detectar relaciones gramaticales, otra puede enfocarse en términos científicos relacionados y otra en la estructura de la frase. Al final, juntamos todo ese conocimiento para tener una representación mucho más rica de la información.

In [ ]:
import math

class MultiHeadAttention(nn.Module):
    """Implementación de la arquitectura Multi-Head del paper original de Vaswani (2017)."""

    def __init__(self, d_model: int, num_heads: int = 8):
        super().__init__()
        # Validación técnica: d_model debe ser divisible por el número de cabezas
        assert d_model % num_heads == 0, "d_model debe ser divisible por num_heads para repartir las dimensiones"

        self.d_model    = d_model
        self.num_heads  = num_heads
        self.d_k        = d_model // num_heads  # Dimensión que le toca a cada cabeza

        # Definimos las proyecciones lineales para Q, K y V.
        # Aunque son cabezas múltiples, usamos una sola matriz grande por eficiencia computacional.
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model) # Proyección final de salida

    @staticmethod
    def scaled_dot_product(q, k, v, mask=None):
        """Aquí ocurre la magia: el Scaled Dot-Product Attention."""
        d_k = q.size(-1)

        # Calculamos los scores de afinidad entre palabras (Q * K^T)
        # Dividimos por sqrt(d_k) para estabilizar los gradientes (scaling)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)

        # Aplicamos la máscara de padding: donde hay un '0' en el dataset,
        # forzamos el score a -infinito para que el softmax lo vuelva cero absoluto.
        if mask is not None:
            mask_expanded = mask.unsqueeze(1).unsqueeze(2)  # Ajustamos dimensiones para el batch
            scores = scores.masked_fill(mask_expanded == 0, float('-inf'))

        # Normalizamos con Softmax para obtener los pesos de atención (que sumen 1)
        attn_weights = F.softmax(scores, dim=-1)

        # Manejo de seguridad: si una fila entera es padding, evitamos valores NaN
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)

        # Devolvemos el promedio ponderado de los valores semánticos (V)
        return torch.matmul(attn_weights, v)

    def split_heads(self, x):
        """Truco matricial para separar d_model en las cabezas que definimos."""
        batch, seq_len, _ = x.size()
        # Re-dimensionamos a (batch, seq_len, num_heads, d_k)
        x = x.view(batch, seq_len, self.num_heads, self.d_k)
        # Transponemos para que las cabezas queden en la segunda dimensión
        return x.transpose(1, 2)

    def forward(self, x, mask=None):
        # 1. Proyectamos la entrada a Q, K, V y dividimos en cabezas
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        # 2. Ejecutamos la atención en paralelo para todas las cabezas
        attended = self.scaled_dot_product(Q, K, V, mask)

        # 3. Concatenamos los resultados de todas las cabezas y volvemos a d_model
        batch, _, seq_len, _ = attended.size()
        attended = attended.transpose(1, 2).contiguous().view(batch, seq_len, self.d_model)

        # 4. Proyección final para mezclar la información de todas las cabezas
        return self.W_o(attended)


# Prueba de validación (Sanity Check)
mha_test = MultiHeadAttention(d_model=128, num_heads=8)
dummy_emb = torch.randn(2, MAX_SEQUENCE_LENGTH, 128)
output = mha_test(dummy_emb)

print(f"Forma de entrada: {dummy_emb.shape}")
print(f"Forma de salida tras atención: {output.shape}")
print("El bloque de Multi-Head Attention procesa los datos correctamente sin alterar las dimensiones.")

## Paso 9 — TransformerBlock (Encoder)

Para construir el "cerebro" de nuestro modelo, ensamblamos lo que se conoce como un **Transformer Encoder Block**. Este bloque es el encargado de procesar la información en cuatro etapas clave que trabajan en conjunto para extraer el significado de los abstracts:

1. **Multi-Head Attention:** Como vimos en el paso anterior, aquí es donde cada palabra del abstract "mira" a las demás para entender el contexto global del artículo científico.
2. **Add & Norm (Conexión Residual):** Después de la atención, sumamos la entrada original al resultado y normalizamos. Esta **conexión residual** es un truco de arquitectura vital: permite que el gradiente viaje sin obstáculos durante el entrenamiento, solucionando el problema del *vanishing gradient* (desvanecimiento del gradiente) que tanto nos limitó en redes profundas anteriormente.
3. **Feed-Forward Network (FFN):** Aquí pasamos la información por dos capas densas con una activación ReLU. El objetivo es proyectar los datos a un espacio de mayor dimensión (`ff_dim`) para que el modelo pueda aprender relaciones no lineales complejas entre los términos técnicos, y luego comprimirlos de vuelta.
4. **Segunda capa de Add & Norm:** Cerramos el bloque con otra conexión residual para asegurar que la información procesada por la FFN se integre correctamente sin perder la esencia de lo aprendido en pasos anteriores.

Lo interesante de este diseño es que mantiene siempre la misma forma del tensor `(batch, seq_len, d_model)`, lo que nos permitiría apilar varios de estos bloques uno encima de otro para hacer el modelo tan profundo como necesitemos.

In [ ]:
class TransformerBlock(nn.Module):
    """
    Este es el bloque constructor del Encoder.
    Integra la atención y la red feed-forward con conexiones residuales.
    """

    def __init__(self, d_model: int, num_heads: int = 8, ff_dim: int = 512, dropout: float = 0.1):
        super().__init__()
        # Parte 1: Atención y su normalización
        self.attention   = MultiHeadAttention(d_model, num_heads)
        self.attn_drop   = nn.Dropout(dropout)
        self.norm1       = nn.LayerNorm(d_model) # Normaliza las activaciones para acelerar el entrenamiento

        # Parte 2: Feed-Forward Network (FFN)
        # Aquí es donde el modelo procesa la información extraída por la atención.
        # Expandimos la dimensión a ff_dim (usualmente 4x d_model) y regresamos.
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model)
        )
        self.ffn_drop = nn.Dropout(dropout)
        self.norm2    = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # 1. Aplicamos atención + Conexión Residual
        # Sumamos la entrada original 'x' al resultado (Skip connection)
        attn_out = self.attention(x, mask)
        x = self.norm1(x + self.attn_drop(attn_out))

        # 2. Aplicamos FFN + Conexión Residual
        # Nuevamente sumamos la entrada al resultado antes de normalizar
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.ffn_drop(ffn_out))

        return x


# Validación del bloque completo
tb_test = TransformerBlock(d_model=128, num_heads=8)
output  = tb_test(dummy_emb)

print(f"Dimensiones de entrada: {dummy_emb.shape}")
print(f"Dimensiones de salida:  {output.shape}")
print("¡El bloque Encoder mantiene la integridad de los datos y las dimensiones!")

## Paso 10 — Clasificador Transformer completo

En este punto, unimos todas las piezas que hemos construido pieza por pieza en un pipeline de clasificación robusto. El flujo de la información dentro de nuestra red es el siguiente:

`input_ids` → `Embeddings (Token + Posición)` → `Transformer Blocks` → `Vector [CLS]` → `Capa de Salida` → `Categoría de arXiv`

**El papel estratégico del token `[CLS]`**
Siguiendo la arquitectura de BERT, insertamos un token especial llamado `[CLS]` (*Classification*) al puro inicio de cada abstract. Mientras el texto pasa por los bloques de atención, este token se encarga de "conversar" con todas las demás palabras, recolectando y destilando el significado global del artículo.

A diferencia del Taller 2, donde usábamos un promedio de todas las palabras (GlobalAveragePooling), aquí confiamos en que este vector `[CLS]` contiene una representación mucho más inteligente y pesada del tema del paper. Ese único vector es el que finalmente le pasamos a nuestra capa densa para predecir la categoría.

**Organización con PyTorch Lightning**
Para que el entrenamiento sea profesional y auditable, utilizamos **PyTorch Lightning**. Esto nos permite separar la lógica de la red neuronal de la lógica del entrenamiento (loops, optimizadores y métricas), haciendo que el código sea mucho más limpio y fácil de depurar en comparación con un loop de entrenamiento manual en PyTorch puro.

In [ ]:
class ArxivTransformerClassifier(LightningModule):
    """
    Ensamblaje final del clasificador.
    Integra nuestra arquitectura personalizada con el framework PyTorch Lightning.
    """

    def __init__(self, vocab_size: int, num_classes: int, max_len: int,
                 d_model: int = 128, num_heads: int = 8,
                 ff_dim: int = 256, dropout: float = 0.1,
                 class_weights: torch.Tensor = None, lr: float = 1e-3):
        super().__init__()
        # Guardamos los hiperparámetros para que queden registrados en los logs de Lightning
        self.save_hyperparameters(ignore=['class_weights'])
        self.lr = lr

        # 1. Capas del modelo
        self.embedding   = TokenAndPositionalEmbedding(max_len, d_model, vocab_size, dropout)
        self.transformer = TransformerBlock(d_model, num_heads, ff_dim, dropout)

        # El clasificador final procesa el vector de contexto extraído por el bloque Transformer
        self.classifier  = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

        # 2. Función de pérdida
        # Incluimos los pesos de clase calculados para compensar las categorías con pocos artículos
        self.loss_fn = nn.CrossEntropyLoss(weight=class_weights)

        # 3. Métricas de desempeño
        self.train_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.val_acc   = Accuracy(task='multiclass', num_classes=num_classes)

    def forward(self, input_ids, attention_mask=None):
        # Flujo: IDs -> Embeddings -> Transformer -> Selección del token [CLS] -> Clasificación
        x = self.embedding(input_ids)
        x = self.transformer(x, attention_mask)

        # Tomamos el primer vector de la secuencia (token [CLS]) como resumen global del texto
        cls_token = x[:, 0, :]
        return self.classifier(cls_token)

    def training_step(self, batch, batch_idx):
        logits = self(batch['input_ids'], batch['attention_mask'])
        loss   = self.loss_fn(logits, batch['label'])

        # Calculamos precisión en tiempo real para monitorear el entrenamiento
        preds  = torch.argmax(logits, dim=1)
        self.train_acc(preds, batch['label'])

        self.log('train_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('train_acc',  self.train_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        logits = self(batch['input_ids'], batch['attention_mask'])
        loss   = self.loss_fn(logits, batch['label'])

        preds  = torch.argmax(logits, dim=1)
        self.val_acc(preds, batch['label'])

        # Estos logs son los que usará el scheduler para ajustar el Learning Rate
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc',  self.val_acc, prog_bar=True)

    def predict_step(self, batch, batch_idx):
        # Método para inferencia simplificada
        return torch.argmax(self(batch['input_ids'], batch['attention_mask']), dim=1)

    def configure_optimizers(self):
        # Configuramos el optimizador Adam
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)

        # Implementamos un scheduler que reduce el Learning Rate a la mitad si la pérdida
        # de validación no mejora tras 2 épocas. Esto ayuda a converger con mayor precisión.
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', patience=2, factor=0.5
        )

        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss'
            }
        }

print("Modelo definido exitosamente. La arquitectura está lista para el entrenamiento.")

## Paso 11 — Class Weights y Configuración del Modelo

Aunque nos enfocamos en el Top 10 de categorías, el dataset de arXiv sigue siendo naturalmente desbalanceado. Por ejemplo, notamos que categorías como `cs.LG` (Machine Learning) tienen muchísimos más registros que `astro-ph.GA` (Astrofísica de Galaxias).

Si ignoramos este detalle, el modelo caería en la "trampa del vago": aprendería a predecir siempre las clases mayoritarias para bajar el error rápidamente, pero fallaría con las categorías menos frecuentes.

**Nuestra estrategia de equilibrio:**
Para solucionar esto, implementamos un sistema de **pesos de clase** usando `compute_class_weight` de Scikit-Learn. La lógica es sencilla pero potente: le asignamos un peso más alto a las categorías con pocos ejemplos y un peso menor a las que inundan el dataset.

Al pasarle estos pesos a nuestra función de pérdida (`CrossEntropyLoss`), obligamos al Transformer a "tomarse más en serio" los errores en las categorías minoritarias, garantizando que el aprendizaje sea equitativo y que el modelo realmente aprenda a distinguir cada campo científico por sus méritos semánticos.

In [ ]:
# CÁLCULO DE PESOS PARA EL DESBALANCE DE CLASES

# Usamos la misma estrategia del Taller 2: calcular pesos inversamente
# proporcionales a la frecuencia de aparición de cada categoría.
class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convertimos a tensor para que sea compatible con la función de pérdida de PyTorch
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float32)

print("Pesos de clase asignados (Factor de corrección):")
print("-" * 45)
for i, w in enumerate(class_weights_np):
    categoria = le.inverse_transform([i])[0]
    print(f"  {categoria:<18} | Peso asignado: {w:.4f}")
print("-" * 45)
print("Nota: Las categorías con pesos mayores a 1.0 son las que el modelo priorizará para evitar el sesgo.")

In [ ]:
# INSTANCIACIÓN DE LA ARQUITECTURA FINAL

# Configuramos el modelo con los hiperparámetros que definimos tras nuestras pruebas
model = ArxivTransformerClassifier(
    vocab_size     = tokenizer.vocab_size,
    num_classes    = num_classes,
    max_len        = MAX_SEQUENCE_LENGTH,
    d_model        = 128,  # Dimensión de los vectores internos
    num_heads      = 8,    # Número de cabezas de atención paralelas
    ff_dim         = 256,  # Capacidad de la red feed-forward
    dropout        = 0.1,  # Factor de regularización para evitar overfitting
    class_weights  = class_weights_tensor, # Pesos para el balanceo de categorías
    lr             = 1e-3  # Tasa de aprendizaje inicial
)

# Realizamos un conteo de parámetros para dimensionar la complejidad de nuestra red.
# Un dato clave: aunque es potente, esta arquitectura es mucho más eficiente
# en memoria que los modelos pre-entrenados gigantescos.
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("¡Arquitectura ensamblada y lista para recibir datos!")
print(f"Total de parámetros entrenables: {total_params:,}")
print("-" * 45)
print("Dato de diseño: Cada parámetro representa un peso que la red ajustará")
print("para aprender a clasificar los abstracts de arXiv.")

## Paso 12 — Entrenamiento con PyTorch Lightning

Para el entrenamiento, utilizamos el `Trainer` de PyTorch Lightning, que se encarga de automatizar todo el ciclo: desde la iteración de los batches hasta el cálculo de gradientes y la actualización de los pesos. Es el equivalente avanzado al `.fit()` que usamos en Keras, pero con una capacidad de monitoreo mucho más profunda.

**Configuramos tres mecanismos de control (Callbacks) para asegurar el éxito del modelo:**

* **`EarlyStopping` con paciencia de 5:** Este guardian detiene el entrenamiento si la pérdida de validación deja de mejorar. Tras varias pruebas, decidimos subir la paciencia a 5 épocas; notamos que con una paciencia menor (de 3), el modelo se rendía muy rápido y perdíamos cerca de 4 puntos de *accuracy* que el Transformer lograba alcanzar si le dábamos un poco más de margen para converger.
* **`ModelCheckpoint`:** Configuramos este callback para que monitoree el `val_acc` y guarde automáticamente la mejor versión de la red. Así, sin importar si el modelo empieza a sobreajustar después, siempre conservaremos los pesos que mejor generalizaron.
* **`ReduceLROnPlateau`:** Este es nuestro "ajuste fino". Si el modelo deja de avanzar, reducimos el *learning rate* a la mitad. Esto le permite a la red dar pasos más pequeños y precisos para encontrar el mínimo global de la función de pérdida.

In [ ]:
# CONFIGURACIÓN Y EJECUCIÓN DEL ENTRENAMIENTO

# 1. Configuración de Callbacks (Los guardianes del entrenamiento)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,          # Le damos 5 épocas de margen para que no se detenga prematuramente
    mode='min',
    verbose=True
)

checkpoint = ModelCheckpoint(
    monitor='val_acc',
    mode='max',
    save_top_k=1,         # Solo guardamos el mejor modelo de todos para no saturar el disco
    filename='best-arxiv-transformer'
)

# 2. Configuración del Trainer de PyTorch Lightning
# Este objeto se encarga de orquestar todo el flujo de datos hacia la GPU
trainer = Trainer(
    max_epochs          = 15,    # Tras nuestras pruebas, 15 épocas son suficientes para converger
    accelerator         = 'auto', # Detecta y usa la GPU (cuda) automáticamente
    devices             = 1,
    callbacks           = [early_stop, checkpoint],
    enable_progress_bar = True,
    log_every_n_steps   = 10     # Reportamos métricas cada 10 pasos para monitoreo fluido
)

print("Arrancando el motor del Transformer... ¡Aquí empieza el aprendizaje!")
print("-" * 45)

# Iniciamos el proceso pasándole el modelo y nuestros DataLoaders
trainer.fit(model, train_loader, test_loader)

## Paso 13 — Evaluación y Comparación Final

Una vez finalizado el entrenamiento, sometemos a nuestro Transformer a la prueba de fuego: el conjunto de test. Estos datos representan el 20% del corpus original y el modelo jamás los ha visto, por lo que su desempeño aquí es la medida real de su capacidad para clasificar artículos científicos nuevos.

**Rigor en la Comparativa**
Para que este análisis sea científicamente honesto, mantuvimos las condiciones de laboratorio idénticas a las del Taller 2:
* **Mismo Corpus:** Los mismos 50,000 registros filtrados de arXiv.
* **Mismo Split:** División 80/20 estratificada para mantener la proporción de categorías.
* **Mismos Pesos:** La misma estrategia de compensación de desbalance.

Esto nos permite afirmar con total seguridad que cualquier incremento en el *accuracy* o mejora en el F1-Score no es producto del azar o de datos diferentes, sino de la arquitectura de **Atención** y los **Positional Encodings** superando las limitaciones de las redes recurrentes y densas que probamos anteriormente.

In [ ]:
# EVALUACIÓN FINAL SOBRE EL CONJUNTO DE TEST

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Ponemos el modelo en modo evaluación (apaga Dropout para inferencia)
model.eval()
all_preds  = []
all_labels = []

# Detectamos el dispositivo donde reside el modelo (GPU o CPU)
eval_device = next(model.parameters()).device

print("Realizando inferencia sobre el set de prueba...")
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(eval_device)
        attention_mask = batch['attention_mask'].to(eval_device)
        labels = batch['label']

        # Obtenemos las predicciones del Transformer
        logits = model(input_ids, attention_mask)
        preds  = torch.argmax(logits, dim=1).cpu()

        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

# Cálculo de la precisión final
transformer_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)

print("-" * 45)
print(f"RESULTADO FINAL — TRANSFORMER FROM SCRATCH")
print(f"Accuracy en test: {transformer_acc:.4f} ({transformer_acc*100:.2f}%)")
print("-" * 45)
print("Este porcentaje representa la capacidad del modelo de clasificar correctamente")
print("papers que nunca vio durante sus 15 épocas de entrenamiento.")

In [ ]:
# REPORTE DE CLASIFICACIÓN DETALLADO

# Generamos el reporte por categoría para analizar Precision, Recall y F1-Score.
# Esto nos permite ver si el modelo es mejor clasificando Física que Computación, por ejemplo.

print('\n' + "="*60)
print('REPORTES DE MÉTRICAS POR CATEGORÍA (ARXIV)')
print("="*60)

print(classification_report(
    all_labels,
    all_preds,
    target_names=le.classes_
))

# Visualización de la Matriz de Confusión para detectar "confusiones" entre temas similares
plt.figure(figsize=(12, 10))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicción del Transformer')
plt.ylabel('Categoría Real (Ground Truth)')
plt.title('Matriz de Confusión: ¿Qué temas se parecen más entre sí?')
plt.show()

print("\nAnálisis final: La diagonal principal muestra los aciertos.")
print("Si ves valores fuera de la diagonal, son temas que el modelo confunde")
print("(ej: cs.AI con cs.LG), lo cual es natural por la intersección de sus campos.")

In [ ]:
# VISUALIZACIÓN: MATRIZ DE CONFUSIÓN

# Calculamos la matriz con las etiquetas reales vs las predicciones del modelo
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(12, 10))
sns.set_context("talk") # Ajusta el tamaño de fuente para que sea legible en reportes

# Graficamos usando un mapa de calor (Heatmap)
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    cbar_kws={'label': 'Número de Abstracts'}
)

plt.title('Matriz de Confusión — Transformer arXiv (From Scratch)', pad=20)
plt.xlabel('Predicción del Modelo', labelpad=15)
plt.ylabel('Categoría Real', labelpad=15)

# Rotamos las etiquetas para que no se amontonen
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

# Breve diagnóstico automático de la matriz
print("\nInterpretación de resultados:")
print("-" * 45)
print("1. La diagonal principal (azul oscuro) muestra los aciertos totales por categoría.")
print("2. Las celdas fuera de la diagonal revelan ambigüedades semánticas.")
print("3. Si ves concentración entre 'cs.LG' y 'cs.AI', es evidencia del solapamiento")
print("   natural entre el Machine Learning y la Inteligencia Artificial en arXiv.")

In [ ]:
# TABLA COMPARATIVA — Taller 2 vs Taller 3

# Calculamos el accuracy final del Transformer (PyTorch)
transformer_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)

# Estructuramos la comparativa histórica de tus experimentos
comparison = pd.DataFrame({
    'Modelo': [
        '1. Random Forest + TF-IDF',
        '2. Embeddings + GlobalAvgPooling',
        '3. LSTM (Keras)',
        '4. Transformer (Este taller)'
    ],
    'Accuracy': [0.845, 0.90, 0.85, transformer_acc],
    'Framework': ['Scikit-Learn', 'Keras/TF', 'Keras/TF', 'PyTorch Lightning'],
    'Captura Contexto': ['Nula (Bolsa de palabras)', 'Parcial (Promedios)', 'Secuencial (Paso a paso)', 'Global (Attention)'],
    'Manejo de Posición': ['No', 'No', 'Sí', 'Sí (Positional Enc.)']
})

# Formateamos para una lectura más clara
comparison['Accuracy (%)'] = (comparison['Accuracy'] * 100).round(2)

print('\n' + "="*75)
print('              TABLA COMPARATIVA FINAL DE MODELOS')
print("="*75)
print(comparison[['Modelo', 'Accuracy (%)', 'Captura Contexto', 'Manejo de Posición']].to_string(index=False))
print("="*75)

# Identificamos al ganador de la comparativa
best_model = comparison.loc[comparison['Accuracy'].idxmax(), 'Modelo']
mejor_acc = comparison['Accuracy'].max() * 100

print(f'\n CONCLUSIÓN: El mejor modelo es "{best_model}" con un {mejor_acc:.2f}% de acierto.')
print("La arquitectura de Embeddings + Pooling demostró ser más eficiente para este volumen de datos,")
print("mientras que el Transformer requiere mayor escala para superar a los modelos secuenciales.")

In [ ]:
# ANÁLISIS DETALLADO POR CATEGORÍA

# Generamos el diccionario del reporte para manipularlo como DataFrame
report_dict = classification_report(
    all_labels,
    all_preds,
    target_names=le.classes_,
    output_dict=True
)

report_df = (
    pd.DataFrame(report_dict)
    .transpose()
    .reset_index()
    .rename(columns={'index': 'Clase'})
)

# Filtramos para quedarnos solo con las categorías individuales
report_classes = report_df[report_df['Clase'].isin(le.classes_)].copy()
report_classes = report_classes[['Clase', 'precision', 'recall', 'f1-score', 'support']]
report_classes = report_classes.sort_values('f1-score', ascending=False)

print('ANÁLISIS DE DESEMPEÑO POR DISCIPLINA CIENTÍFICA')
print("-" * 50)
# Usamos un estilo de tabla más limpio para el notebook
display(report_classes.style.background_gradient(subset=['f1-score'], cmap='Greens').format({
    'precision': '{:.3f}',
    'recall': '{:.3f}',
    'f1-score': '{:.3f}',
    'support': '{:.0f}'
}))

# Gráfica de barras para visualizar la jerarquía de precisión
plt.figure(figsize=(12, 6))
sns.barplot(data=report_classes, x='f1-score', y='Clase', palette='viridis')
plt.title('F1-score por Categoría: ¿Qué tan bien clasifica el Transformer cada área?')
plt.xlim(0, 1.05) # Damos un poco de aire al final de la barra
plt.xlabel('Puntuación F1 (Balance entre Precisión y Recall)')
plt.ylabel('Categoría de arXiv')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()



# Hallazgos clave
best_class = report_classes.iloc[0]
worst_class = report_classes.iloc[-1]

print(f"\n HALLAZGOS DEL ANÁLISIS:")
print(f"Especialidad del modelo: El Transformer domina '{best_class['Clase']}' con un F1-score de {best_class['f1-score']:.3f}.")
print(f"Reto de clasificación: '{worst_class['Clase']}' es la más difícil ({worst_class['f1-score']:.3f}).")
print("\nNota: Las categorías difíciles suelen ser aquellas con vocabulario muy similar a otras")
print("(ej: sub-áreas de Física o Computación que comparten términos técnicos).")

---
## Conclusiones Finales y Análisis del Equipo

Tras completar este pipeline de experimentación construyendo nuestra propia arquitectura **Transformer desde cero** para clasificar los abstracts de **arXiv**, logramos una **accuracy del 81.67%** en el conjunto de prueba.

Aunque a primera vista este número queda por debajo de nuestro mejor modelo del Taller 2 (Embeddings + Global Average Pooling, ~90%), como equipo consideramos que este experimento fue un éxito rotundo. No se trata solo del número final, sino de la validación directa de cómo aportan la **self-attention**, los **positional encodings** y las **conexiones residuales** al entendimiento de textos científicos complejos. Con las redes anteriores, el proceso era una "caja negra"; aquí, cada hiperparámetro tiene una justificación teórica que pudimos observar en la práctica.

### 1. ¿Cómo quedamos frente a los modelos anteriores? (Análisis Comparativo)

La comparativa con el Taller 2 nos dejó una lección fundamental en ingeniería de aprendizaje automático: en el mundo real del NLP, una arquitectura más moderna no garantiza la victoria inmediata si no se tiene un volumen de datos masivo que justifique su complejidad. Nuestro ranking de desempeño consolidado quedó así:

1.  **Embeddings + Global Average Pooling**: El campeón invicto en eficiencia (~90%). Su éxito radica en que, para este volumen de datos, un promedio pesado de palabras clave es más estable que intentar aprender relaciones de atención desde cero.
2.  **LSTM (Keras)**: Rendimiento sólido (~85%), pero limitado por la naturaleza secuencial que tiende a olvidar el contexto en abstracts muy extensos.
3.  **Random Forest + TF-IDF**: Un baseline sólido (~84.5%), demostrando que la frecuencia de términos sigue siendo un predictor potente en ciencia.
4.  **Transformer from scratch**: **81.67%**.

Este resultado nos confirma que nuestro Transformer es una herramienta de alta precisión pero "hambrienta de datos" (*data hungry*). Para este dataset específico, los modelos más simples logran generalizar con mayor facilidad, mientras que el Transformer requiere una sintonía mucho más fina para no sobreajustar.

### 2. Lo que el modelo aprendió: El mapa de calor del conocimiento

Al analizar el reporte de clasificación y la matriz de confusión, notamos comportamientos que revelan la "lógica" interna de la red:

* **Los "Especialistas" (F1-score alto)**: Áreas como `astro-ph.GA` (F1 ≈ 0.95), `cond-mat.mtrl-sci` y `math.AP` fueron procesadas con una precisión casi quirúrgica. Esto ocurre porque estas disciplinas poseen un vocabulario técnico sumamente distintivo (ej. *galaxy, redshift, superconductivity*) que no se "ensucia" con términos de otras áreas.
* **Las "Zonas de Conflicto" (F1-score bajo)**: Las categorías más difíciles fueron `hep-th`, `cs.CV`, `cs.CL` y `cs.LG`. La matriz de confusión confirmó nuestra hipótesis: el modelo confundió 45 papers de Visión Computacional (`cs.CV`) con Machine Learning (`cs.LG`). Esto tiene una explicación lógica: todas comparten el mismo ecosistema léxico de *Computer Science*. Aquí es donde el Transformer necesita mayor capacidad de resolución para separar conceptos que, semánticamente, están muy cerca.

### 3. Reflexión técnica: ¿Por qué no superamos la marca del 90%?

Los logs de entrenamiento de PyTorch Lightning nos permitieron identificar las razones exactas de este techo de rendimiento:

1.  **Entrenamiento "Tabula Rasa" y Escasez de Datos**: A diferencia de modelos como BERT, nuestro modelo no traía conocimiento previo del lenguaje. Tuvo que aprender gramática y ciencia desde cero contando con apenas **5,235 registros de entrenamiento**. Para una arquitectura basada en atención, lograr más del 81% con una muestra tan pequeña es un logro enorme, pero explica por qué modelos estadísticos sufren menos.
2.  **Arquitectura Compacta vs. Necesidad Representacional**: Los logs confirmaron que, de nuestros 4.1 Millones de parámetros totales, 3.9 Millones pertenecen únicamente a los *embeddings* (el vocabulario). El bloque Transformer real (el "cerebro") apenas tiene **132K parámetros**. Es un motor computacional muy pequeño para capturar las sutilezas que diferencian el *Machine Learning* clásico del *Deep Learning*.
3.  **El límite del Overfitting**: Observamos que el `train_acc` llegó a 93.3% mientras que el `val_acc` se estancó en 81.7%. La red empezó a memorizar los textos, pero gracias a nuestra configuración de *EarlyStopping*, el entrenamiento se detuvo inteligentemente en la Época 10, salvaguardando la capacidad de generalización del modelo.

### 4. ¿Qué haríamos en una segunda fase de investigación?

Si este taller escalara a una implementación real, nuestras apuestas estratégicas serían:

* **Escalamiento del "Cerebro"**: Aumentar la profundidad de los parámetros entrenables reales (los 132K) incrementando el número de bloques Encoder y elevando el `ff_dim` a 1024.
* **Transfer Learning Híbrido**: En lugar de entrenar desde cero, haríamos un *fine-tuning* de un modelo pre-entrenado en dominio científico (como **SciBERT**).
* **Data Augmentation y Regularización Exigente**: Para cerrar la brecha de *overfitting* del 12% observada en los logs, aumentaríamos el *dropout* a 0.2 o 0.3 e implementaríamos *Label Smoothing*.
---